# Deep Learning for Biological Modeling
### SIAM Life Sciences 2026 — Example 2: Equation Learning for ODEs

This notebook demonstrates **symbolic regression** applied to equation learning for
dynamical systems — recovering the right-hand-side of an ODE from trajectory data.

We use the **Lotka–Volterra predator–prey model** as our target system throughout.

**Structure:**
1. Background: the equation-learning problem and how DSO differs from SINDy
2. The DSO framework: expression trees, RNN policy, and REINFORCE (full math)
3. Mini-DSO warm-up: a minimal implementation for logistic growth (1 variable)
4. The Lotka–Volterra model: trajectories and phase portrait
5. Extended Mini-DSO: recovering the full predator–prey equations (2 variables)
6. Symbolic regression on noisy data — the derivative estimation challenge
7. Conceptual comparison: DSO vs. SINDy
8. *(Optional)* SINDy on the same data

**Dependencies:** `numpy`, `scipy`, `matplotlib`, `tensorflow` — all pre-installed in Colab.
No additional installs required.

Run in **Google Colab**: execute the setup cell first, then run cells in order.


## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import savgol_filter
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)
print("All dependencies loaded — no installs required.")


/Users/suzanne/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow: 2.20.0
All dependencies loaded — no installs required.


---
## 1. Background: Equation Learning from Trajectory Data

### The problem

Suppose we observe a trajectory $\mathbf{x}(t) \in \mathbb{R}^d$ of a dynamical system and
hypothesise that it obeys an autonomous ODE:

$$\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}), \qquad \mathbf{x}(0) = \mathbf{x}_0$$

The **equation learning** (or *system identification*) task is: given measurements
$\{\mathbf{x}(t_i)\}_{i=1}^N$, recover $\mathbf{f}$ in a form that is interpretable — ideally
as a closed-form symbolic expression.

This is harder than function approximation: we want not just a neural network that
approximates $\mathbf{f}$, but the *symbolic* form (e.g., $\alpha x - \beta xy$) with
identifiable parameters.

---

### SINDy (a brief reminder)

You are likely familiar with **SINDy** (Brunton, Proctor & Kutz 2016) and its weak-form
extensions. SINDy parameterises $\mathbf{f}$ as a sparse linear combination of a
pre-specified function library $\Theta(\mathbf{x})$:

$$\dot{\mathbf{x}} \approx \Theta(\mathbf{x})\,\Xi, \qquad \Theta(\mathbf{x}) = [\mathbf{1},\; \mathbf{x},\; \mathbf{x}^2,\; \mathbf{x} \otimes \mathbf{x},\; \sin(\mathbf{x}),\; \ldots]$$

and solves a sparse regression:

$$\hat{\Xi} = \operatorname*{arg\,min}_{\Xi}\|\dot{X} - \Theta(X)\Xi\|_F^2 + \lambda\|\Xi\|_0$$

The **weak form** (Messenger & Bortz 2021; Reinbold et al. 2020) avoids noise amplification from
numerical differentiation by projecting onto test functions $\phi_k$:

$$\int \phi_k(t)\, \dot{\mathbf{x}}\, dt = -\int \dot{\phi}_k(t)\, \mathbf{x}\, dt \approx \int \phi_k(t)\, \Theta(\mathbf{x})\Xi\, dt$$

**The key limitation of SINDy:** the library $\Theta$ must be pre-specified. If the
true dynamics involve a function *not* in the library, it cannot be discovered. Library
design requires domain knowledge and grows combinatorially for high-dimensional systems.

---

### Expression-tree methods: a different representation

**DSO** (Petersen et al. 2021) and **PySR** (Cranmer 2023) instead represent $f$ as a
**symbolic expression tree**, where:

- *Leaf nodes* are variables ($x$, $y$) or constants ($c$, fitted by regression)
- *Internal nodes* are mathematical operators ($+$, $-$, $*$, $/$, $\sin$, ...)
- The tree encodes a symbolic expression without requiring a pre-specified library

Example: the tree for $\alpha x - \beta xy$ in **prefix notation** (operator before operands):

```
−  (root)
├── *
│   ├── α
│   └── x
└── *
    ├── β
    └── *
        ├── x
        └── y
```

This is equivalent to the token sequence `[ −, *, c, x, *, c, *, x, y ]`.

The **search problem** is now over expression trees rather than over coefficient vectors.
DSO solves this with a **recurrent neural network + reinforcement learning** (Section 2).
PySR solves it with evolutionary (genetic) algorithms.

The key advantage over SINDy: **no library is required**. The algorithm discovers which
operators and variables to combine, including nested compositions (e.g., $\sin(x^2)$)
that would need to be explicitly added to a SINDy library.


---
## 2. The DSO Framework: Expression Trees + RNN + REINFORCE

*(Petersen, Larma, Mundhenk, Santiago, Kim & Kim, 2021)*

### 2.1 Programs as sequences

A symbolic expression in prefix notation is a token sequence
$\tau = (a_1, a_2, \ldots, a_T)$ drawn from a finite vocabulary
$V = V_{\text{ops}} \cup V_{\text{leaves}}$.
Any valid prefix sequence encodes a unique expression tree.

### 2.2 The policy: an RNN over tokens

A recurrent neural network (in the original paper, an LSTM) parameterises a
*distribution over programs*. At each step $t$, conditioned on the previously sampled
tokens $a_{1:t-1}$:

$$\pi_\theta(a_t \mid a_{1:t-1}) = \text{softmax}\bigl(W_{\text{out}}\, h_t + b_{\text{out}}\bigr)$$

$$h_t = \text{GRU}\bigl(e(a_{t-1}),\; h_{t-1}\bigr)$$

where $e(a_{t-1})$ is an embedding of the previous token. The joint probability of a program is:

$$\pi_\theta(\tau) = \prod_{t=1}^{T} \pi_\theta(a_t \mid a_{1:t-1})$$

### 2.3 Reward

Given a candidate expression $\tau$ and training data $(\mathbf{X}, \dot{\mathbf{x}})$,
fit any constants in $\tau$ by least squares, then evaluate:

$$R(\tau) = 1 - \frac{\|\dot{\mathbf{x}} - \hat{f}_\tau(\mathbf{X})\|^2}{\|\dot{\mathbf{x}} - \bar{\dot{\mathbf{x}}}\|^2} \in [0, 1]$$

This is the coefficient of determination $R^2$ — higher is better. Programs that
fail to parse receive $R(\tau) = 0$.

### 2.4 The REINFORCE objective

The goal is to maximise expected reward:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$$

The **policy gradient theorem** (Williams 1992) gives the gradient:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[R(\tau)\,\nabla_\theta \log \pi_\theta(\tau)\right]$$

Since $\log \pi_\theta(\tau) = \sum_t \log \pi_\theta(a_t \mid a_{1:t-1})$, this becomes:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[R(\tau) \sum_{t=1}^{T} \nabla_\theta \log \pi_\theta(a_t \mid a_{1:t-1})\right]$$

In practice, we draw a batch of $B$ programs, subtract a **baseline** $b$ (exponential moving
average of recent rewards) to reduce gradient variance, and update:

$$\theta \leftarrow \theta + \eta \cdot \frac{1}{B}\sum_{i=1}^{B} \bigl(R(\tau^{(i)}) - b\bigr)\sum_{t=1}^{T}\nabla_\theta \log \pi_\theta(a_t^{(i)} \mid a_{1:t-1}^{(i)})$$

The Petersen et al. paper further adds **risk-seeking** refinements (using only the top
percentile of programs) and an **entropy regularisation** term $\lambda H(\pi_\theta)$ to
encourage exploration. For our tutorial, vanilla REINFORCE with a baseline is sufficient.

### 2.5 Comparison to SINDy — in a sentence

| | SINDy | DSO |
|---|-------|-----|
| Representation | Sparse vector $\Xi$ over pre-specified $\Theta$ | Symbolic expression tree |
| Search / optimisation | Convex sparse regression (LASSO/STLSQ) | Policy gradient RL (first-order, non-convex) |
| Library required? | Yes | No |
| Nested expressions | Limited | Natural |
| Noise handling | Weak-form: very good | Depends on implementation |
| Compute | Seconds | Minutes–hours |
| Convergence guarantees | Sparse recovery theory (RIP conditions) | Heuristic |


---
## 3. Mini-DSO: A From-Scratch Implementation

Before using PySR on Lotka–Volterra, we implement a minimal DSO to make the
mechanism tangible. This is deliberately simple:

- **Vocabulary:** $V = \{+,\, *,\, x,\, c\}$ (4 tokens)
- **Target:** recover the growth term of a logistic ODE, $\dot{x} = x(1 - x) = x - x^2$,
  from synthetic data. Both terms have the form $c_i \cdot x^{n_i}$, so our vocabulary suffices.
- **Policy:** a GRU network (64 hidden units) that samples token sequences
- **Reward:** $R^2$ after fitting a single scale constant by least squares
- **Update:** REINFORCE with exponential moving average baseline

This takes ~1 minute on CPU. Watch the reported best expression converge toward `(c * x)` and `(c * (x * x))`.


In [2]:
# ─── Vocabulary ────────────────────────────────────────────────────────────
# Token indices:  0='+'  1='*'  2='x'  3='c'  4=<START> (input-only)
TOKENS   = ['+', '*', 'x', 'c']
OPS      = {0, 1}           # binary operators
N_TOK    = len(TOKENS)
MAX_LEN  = 9               # maximum sequence length sampled


In [3]:
# ─── Prefix expression evaluator ───────────────────────────────────────────
def eval_prefix(token_list, x_vals):
    """
    Parse a prefix token sequence and evaluate on x_vals.
    Returns (result_array, n_tokens_consumed).
    Constants 'c' evaluate to 1.0 (scale fitted externally).
    """
    idx = [0]

    def _parse():
        if idx[0] >= len(token_list):
            return None
        tok = token_list[idx[0]]; idx[0] += 1
        if tok == 0:             # '+'
            l, r = _parse(), _parse()
            return None if (l is None or r is None) else l + r
        elif tok == 1:           # '*'
            l, r = _parse(), _parse()
            return None if (l is None or r is None) else l * r
        elif tok == 2:           # 'x'
            return x_vals.copy()
        elif tok == 3:           # 'c'  (placeholder = 1.0)
            return np.ones_like(x_vals)
        return None

    result = _parse()
    return result, idx[0]


def reward(token_list, x_vals, y_vals):
    """Fit scale constant c by least squares; return R², c, tokens_used."""
    pred, n_used = eval_prefix(token_list, x_vals)
    if pred is None or not np.all(np.isfinite(pred)):
        return 0.0, 1.0, 0
    denom = np.dot(pred, pred) + 1e-10
    c_opt = np.dot(y_vals, pred) / denom
    pred_fit = c_opt * pred
    ss_res = np.sum((y_vals - pred_fit)**2)
    ss_tot = np.sum((y_vals - np.mean(y_vals))**2) + 1e-10
    return float(max(0.0, 1.0 - ss_res / ss_tot)), float(c_opt), n_used


def tokens_to_str(tlist):
    """Pretty-print a prefix token list as an infix expression."""
    idx = [0]
    def _fmt():
        if idx[0] >= len(tlist): return '?'
        tok = tlist[idx[0]]; idx[0] += 1
        if tok in OPS:
            l, r = _fmt(), _fmt()
            return f'({l} {TOKENS[tok]} {r})'
        return TOKENS[tok]
    return _fmt()


In [4]:
# ─── GRU Policy Network ────────────────────────────────────────────────────
class GRUPolicy(tf.keras.Model):
    """
    At each step t, takes the previous token (or a START token) and
    the current hidden state, and outputs a log-probability distribution
    over the next token.

    n_tokens: vocabulary size (4 for warm-up, 6 for Lotka–Volterra)
    hidden  : GRU hidden dimension
    """
    def __init__(self, n_tokens=N_TOK, hidden=64):
        super().__init__()
        self.n_tokens = n_tokens   # stored for use in step()
        self.hidden   = hidden
        self.cell     = layers.GRUCell(hidden)
        self.head     = layers.Dense(n_tokens)

    def init_state(self):
        return tf.zeros((1, self.hidden))

    def step(self, prev_tok_idx, state):
        """Returns (log_probs of shape (n_tokens,), new_state)."""
        x = tf.one_hot([[prev_tok_idx]], self.n_tokens + 1)   # +1 for START
        out, new_state = self.cell(x[:, 0, :], [state])
        log_probs = tf.nn.log_softmax(self.head(out))[0]
        return log_probs, new_state[0]


In [5]:
# ─── Sampling ──────────────────────────────────────────────────────────────
def sample_program(policy):
    """
    Sample one token sequence from the policy (no validity constraints —
    invalid expressions are penalised by R²=0).
    """
    state = policy.init_state()
    prev  = N_TOK            # START token
    toks  = []
    for _ in range(MAX_LEN):
        lp, state = policy.step(prev, state)
        tok = int(tf.random.categorical(lp[tf.newaxis], 1)[0, 0].numpy())
        toks.append(tok)
        prev = tok
    return toks


def sample_batch(policy, batch_size):
    return [sample_program(policy) for _ in range(batch_size)]


In [6]:
# ─── REINFORCE training loop ───────────────────────────────────────────────
def train_mini_dso(x_vals, y_vals, n_iter=400, batch_size=64, lr=3e-3):
    policy   = GRUPolicy()
    opt      = tf.keras.optimizers.Adam(lr)
    baseline = 0.0
    beta     = 0.9        # EMA decay for baseline

    best_r2, best_prog, best_c = 0.0, None, 1.0
    history = []

    for it in range(n_iter):
        programs = sample_batch(policy, batch_size)

        # ── Compute rewards (outside tape: numpy only) ──────────────────
        rews, cs, n_useds = [], [], []
        for prog in programs:
            r, c, n = reward(prog, x_vals, y_vals)
            rews.append(r); cs.append(c); n_useds.append(n)
        rews_np = np.array(rews, dtype=np.float32)

        # Update baseline (EMA) and compute advantages
        baseline = beta * baseline + (1 - beta) * rews_np.mean()
        advs     = rews_np - baseline

        # Track best
        bi = int(np.argmax(rews_np))
        if rews_np[bi] > best_r2:
            best_r2, best_prog, best_c = float(rews_np[bi]), programs[bi][:], cs[bi]

        # ── REINFORCE gradient update (inside tape) ─────────────────────
        with tf.GradientTape() as tape:
            loss = tf.constant(0.0)
            for prog, adv, n in zip(programs, advs, n_useds):
                if n == 0: continue
                # Re-run forward pass through policy so gradients are tracked
                state = policy.init_state()
                prev  = N_TOK
                lp_sum = tf.constant(0.0)
                for tok in prog[:n]:
                    lp, state = policy.step(prev, state)
                    lp_sum = lp_sum + lp[tok]
                    prev = tok
                loss = loss - tf.cast(adv, tf.float32) * lp_sum
            loss = loss / batch_size

        grads = tape.gradient(loss, policy.trainable_variables)
        opt.apply_gradients(zip(grads, policy.trainable_variables))
        history.append(best_r2)

        if (it + 1) % 100 == 0:
            print(f"  Iter {it+1:4d} | best R²={best_r2:.4f} | "
                  f"expr: {tokens_to_str(best_prog)} × {best_c:.4f}")

    return best_prog, best_r2, best_c, history


In [ ]:
# ─── Target: logistic growth  dx/dt = x - x²  ─────────────────────────────
# We run two separate DSO searches: one for the +x term, one for the -x² term.
# (In practice, DSO searches for the full RHS; we split here for clarity.)

x_demo = np.linspace(0.05, 0.95, 200).astype(np.float32)
dxdt_pos = x_demo.copy()          # the  +x  component
dxdt_neg = -(x_demo ** 2)         # the  -x² component

print("=" * 60)
print("Searching for  +c·x  (should find: c ≈ +1.0)")
prog_pos, r2_pos, c_pos, hist_pos = train_mini_dso(x_demo, dxdt_pos, n_iter=400)
print(f"\nBest: {tokens_to_str(prog_pos)} × {c_pos:.4f}  (R²={r2_pos:.4f})")

print("\n" + "=" * 60)
print("Searching for  -c·x²  (should find structure x*x, c ≈ -1.0)")
prog_neg, r2_neg, c_neg, hist_neg = train_mini_dso(x_demo, dxdt_neg, n_iter=400)
print(f"\nBest: {tokens_to_str(prog_neg)} × {c_neg:.4f}  (R²={r2_neg:.4f})")


Searching for  +c·x  (should find: c ≈ +1.0)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, hist, title in zip(axes, [hist_pos, hist_neg], ["+c·x", "c·x²"]):
    ax.plot(hist)
    ax.set_xlabel("Iteration"); ax.set_ylabel("Best R²")
    ax.set_title(f"Mini-DSO convergence — target: {title}")
    ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()
print("The policy learns to sample increasingly good expressions as training progresses.")


---
## 4. The Lotka–Volterra Predator–Prey Model

The classical Lotka–Volterra system describes coupled prey ($x$) and predator ($y$) populations:

$$\dot{x} = \alpha x - \beta xy$$
$$\dot{y} = \delta xy - \gamma y$$

with parameters:
- $\alpha = 1.0$: per-capita prey birth rate
- $\beta = 0.1$: predation rate
- $\delta = 0.075$: predator reproduction rate per predation event
- $\gamma = 1.5$: per-capita predator death rate

These equations have two types of terms:
- **Linear**: $\alpha x$ and $-\gamma y$ — single-species growth or decay
- **Quadratic interaction**: $-\beta xy$ and $+\delta xy$ — the predator–prey coupling

In prefix notation (vocabulary $\{-, *, c, x, y\}$):
$$\dot{x}: \quad \texttt{- * c x * c * x y}$$
$$\dot{y}: \quad \texttt{- * c * x y * c y}$$

Both are 9-token sequences — well within reach of the expression-tree search.


In [ ]:
# Lotka–Volterra parameters
alpha = 1.0;  beta  = 0.1
delta = 0.075; gamma = 1.5

def lotka_volterra(t, state):
    x, y = state
    return [alpha*x - beta*x*y,
            delta*x*y - gamma*y]

# Simulate: one long trajectory, densely sampled
t_span = (0, 40)
t_eval = np.linspace(*t_span, 2000)
sol = solve_ivp(lotka_volterra, t_span, y0=[10.0, 5.0],
                t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)

x_traj = sol.y[0]   # prey
y_traj = sol.y[1]   # predator
t_traj = sol.t

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t_traj, x_traj, label='Prey  x(t)', color='steelblue')
axes[0].plot(t_traj, y_traj, label='Predator y(t)', color='darkorange')
axes[0].set_xlabel('Time'); axes[0].set_ylabel('Population')
axes[0].set_title('Lotka–Volterra trajectories'); axes[0].legend()

axes[1].plot(x_traj, y_traj, color='mediumseagreen', lw=0.8)
axes[1].set_xlabel('Prey x'); axes[1].set_ylabel('Predator y')
axes[1].set_title('Phase portrait')
plt.tight_layout(); plt.show()


---
## 5. Extended Mini-DSO: Recovering Lotka–Volterra Equations

We now apply the DSO mechanism directly to the Lotka–Volterra problem.
The vocabulary is extended to two variables and subtraction:

$$V = \{+,\; -,\; *,\; x,\; y,\; c\}$$

The Lotka–Volterra equations each contain **two constants**:

$$\dot{x} = \underbrace{\alpha}_{c_1} \cdot x \;-\; \underbrace{\beta}_{c_2} \cdot x y \qquad \dot{y} = \underbrace{\delta}_{c_1} \cdot x y \;-\; \underbrace{\gamma}_{c_2} \cdot y$$

In the expression tree, $c_1$ and $c_2$ appear as separate leaf nodes.
To fit them simultaneously, we build a **design matrix** $\Phi \in \mathbb{R}^{N \times n_c}$
where column $k$ is the expression evaluated with the $k$-th constant leaf set to 1
and all others to 0. Then:

$$\hat{\mathbf{c}} = \arg\min_{\mathbf{c}} \|\mathbf{d} - \Phi\,\mathbf{c}\|^2
= (\Phi^\top \Phi)^{-1} \Phi^\top \mathbf{d}$$

Two improvements over the warm-up DSO help convergence on this harder problem:

- **Validity constraints**: at each step, if there are not enough token slots left to
  accommodate another binary operator, operator tokens are masked out. Every sampled
  sequence is guaranteed to be a syntactically valid expression.
- **Entropy regularisation**: the term $\lambda H(\pi_\theta)$ is added to the objective,
  encouraging the policy to keep exploring rather than collapsing onto a local optimum.


In [ ]:
# True derivatives (no noise)
dxdt_true = alpha*x_traj - beta*x_traj*y_traj
dydt_true = delta*x_traj*y_traj - gamma*y_traj

# Subsample to 500 points for speed (still more than enough)
idx = np.linspace(0, len(t_traj)-1, 500, dtype=int)
X_clean = np.column_stack([x_traj[idx], y_traj[idx]])
dxdt_c  = dxdt_true[idx]
dydt_c  = dydt_true[idx]

print(f"Training samples: {len(idx)}")
print(f"x range: [{X_clean[:,0].min():.2f}, {X_clean[:,0].max():.2f}]")
print(f"y range: [{X_clean[:,1].min():.2f}, {X_clean[:,1].max():.2f}]")


In [ ]:
# ─── Extended vocabulary ────────────────────────────────────────────────────
# Indices: 0='+'  1='-'  2='*'  3='x'  4='y'  5='c'  6=<START> (input only)
LV_TOKENS  = ['+', '-', '*', 'x', 'y', 'c']
LV_OPS     = {0, 1, 2}        # binary operators
LV_N_TOK   = 6
LV_MAX_LEN = 11               # max sequence length (9-token targets fit comfortably)

# ─── Multi-constant evaluator ────────────────────────────────────────────────
def _eval_kth_const(tokens, xv, yv, k):
    """
    Evaluate the prefix expression tree with the k-th constant leaf = 1.0
    and all other constant leaves = 0.0.
    Returns a numpy array of shape (N,), or None if the expression is invalid.
    """
    idx = [0]; c_count = [0]

    def _parse():
        if idx[0] >= len(tokens): return None
        tok = tokens[idx[0]]; idx[0] += 1
        if tok == 0:                       # '+'
            l, r = _parse(), _parse()
            return None if (l is None or r is None) else l + r
        elif tok == 1:                     # '-'
            l, r = _parse(), _parse()
            return None if (l is None or r is None) else l - r
        elif tok == 2:                     # '*'
            l, r = _parse(), _parse()
            return None if (l is None or r is None) else l * r
        elif tok == 3: return xv.copy()   # 'x'
        elif tok == 4: return yv.copy()   # 'y'
        elif tok == 5:                     # 'c'
            val = 1.0 if c_count[0] == k else 0.0
            c_count[0] += 1
            return val * np.ones_like(xv)
        return None

    return _parse()


def fit_and_score_lv(tokens, xv, yv, target):
    """
    Fit all constant leaves by linear least squares; return (R^2, c_vals).

    Design matrix: Phi[:, k] = expression evaluated with c_k=1, others=0.
    Fit: c_hat = argmin ||target - Phi @ c||^2   (standard lstsq)
    """
    n_c = sum(1 for t in tokens if t == 5)
    if n_c == 0:
        return 0.0, np.array([])

    xd = xv.astype(np.float64); yd = yv.astype(np.float64)
    Phi = np.zeros((len(xv), n_c), dtype=np.float64)
    for k in range(n_c):
        col = _eval_kth_const(tokens, xd, yd, k)
        if col is None or not np.all(np.isfinite(col)):
            return 0.0, np.zeros(n_c)
        Phi[:, k] = col

    try:
        c_opt, _, _, _ = np.linalg.lstsq(Phi, target.astype(np.float64), rcond=None)
        pred  = Phi @ c_opt
        ss_res = np.sum((target - pred)**2)
        ss_tot = np.sum((target - np.mean(target))**2) + 1e-10
        return float(max(0.0, 1.0 - ss_res / ss_tot)), c_opt
    except Exception:
        return 0.0, np.zeros(n_c)


def lv_tokens_to_str(tokens, c_vals=None):
    """Infix-format a prefix token list; substitute fitted values when provided."""
    idx = [0]; c_count = [0]
    greek = ['α', 'β', 'γ', 'δ', 'ε', 'ζ']

    def _fmt():
        if idx[0] >= len(tokens): return '?'
        tok = tokens[idx[0]]; idx[0] += 1
        if tok in LV_OPS:
            l, r = _fmt(), _fmt()
            return f'({l} {LV_TOKENS[tok]} {r})'
        elif tok == 5:
            if c_vals is not None and c_count[0] < len(c_vals):
                val = float(c_vals[c_count[0]]); c_count[0] += 1
                return f'{val:.4f}'
            name = greek[c_count[0] % len(greek)]; c_count[0] += 1
            return name
        return LV_TOKENS[tok]
    return _fmt()


In [ ]:
# ─── Validity-constrained sampler ────────────────────────────────────────────
def _validity_mask(needed, remaining_after):
    """
    Build a log-scale validity mask for the token distribution.

    'needed'          = token slots still required to complete the expression.
    'remaining_after' = LV_MAX_LEN - step - 1.

    Rule: picking a binary operator at this step would set needed -> needed+1.
    If remaining_after <= needed, there is not enough room for another operator
    (we would need at least needed+1 more tokens), so we force a leaf.
    """
    if remaining_after <= needed:
        return np.array([-1e9 if i in LV_OPS else 0.0
                         for i in range(LV_N_TOK)], dtype=np.float32)
    return np.zeros(LV_N_TOK, dtype=np.float32)


def sample_valid_lv(policy):
    """Sample one syntactically valid expression from the LV policy."""
    state  = policy.init_state()
    prev   = LV_N_TOK          # START token (index 6)
    tokens = []
    needed = 1                 # slots needed to complete the expression

    for step in range(LV_MAX_LEN):
        lp, state = policy.step(prev, state)
        remaining_after = LV_MAX_LEN - step - 1

        mask        = _validity_mask(needed, remaining_after)
        masked_lp   = lp + tf.constant(mask)
        tok = int(tf.random.categorical(masked_lp[tf.newaxis], 1)[0, 0].numpy())

        tokens.append(tok)
        needed = needed + 1 if tok in LV_OPS else needed - 1
        prev   = tok
        if needed == 0:
            break   # expression complete

    return tokens


In [ ]:
# ─── REINFORCE training loop with entropy regularisation ────────────────────
def train_lv_dso(xv, yv, target, n_iter=800, batch_size=128,
                 lr=3e-3, ent_weight=0.005):
    """
    Run DSO on one ODE component (dx/dt or dy/dt).

    xv, yv, target : numpy arrays of shape (N,)
    Returns: (best_tokens, best_r2, best_c_vals, r2_history)
    """
    policy   = GRUPolicy(n_tokens=LV_N_TOK, hidden=64)
    opt      = tf.keras.optimizers.Adam(lr)
    baseline = 0.0
    beta     = 0.9                    # EMA decay for baseline

    best_r2, best_prog, best_cs = 0.0, [5], np.array([1.0])
    history = []

    for it in range(n_iter):
        # ── Sample batch of valid programs ──────────────────────────────
        programs = [sample_valid_lv(policy) for _ in range(batch_size)]

        # ── Compute rewards (numpy, outside tape) ────────────────────────
        rews = []; all_cs = []
        for prog in programs:
            r, cs = fit_and_score_lv(prog, xv, yv, target)
            rews.append(r); all_cs.append(cs)
        rews_np = np.array(rews, dtype=np.float32)

        baseline = beta * baseline + (1 - beta) * float(rews_np.mean())
        advs     = rews_np - baseline

        bi = int(np.argmax(rews_np))
        if rews_np[bi] > best_r2:
            best_r2 = float(rews_np[bi])
            best_prog = programs[bi][:]
            best_cs   = all_cs[bi].copy()

        # ── REINFORCE gradient update ────────────────────────────────────
        with tf.GradientTape() as tape:
            pol_loss  = tf.constant(0.0)
            ent_total = tf.constant(0.0)

            for prog, adv in zip(programs, advs):
                state  = policy.init_state()
                prev   = LV_N_TOK
                lp_sum = tf.constant(0.0)
                needed = 1

                for step, tok in enumerate(prog):
                    lp, state = policy.step(prev, state)
                    rem = LV_MAX_LEN - step - 1

                    mask_tf = tf.constant(_validity_mask(needed, rem))
                    masked  = lp + mask_tf
                    log_pi  = tf.nn.log_softmax(masked)
                    pi      = tf.exp(log_pi)

                    lp_sum    = lp_sum + log_pi[tok]
                    ent_total = ent_total - tf.reduce_sum(pi * log_pi)

                    needed = needed + 1 if tok in LV_OPS else needed - 1
                    prev   = tok

                pol_loss = pol_loss - tf.cast(adv, tf.float32) * lp_sum

            total_loss = (pol_loss / batch_size) - ent_weight * (ent_total / batch_size)

        grads = tape.gradient(total_loss, policy.trainable_variables)
        opt.apply_gradients(zip(grads, policy.trainable_variables))
        history.append(best_r2)

        if (it + 1) % 100 == 0:
            print(f"  Iter {it+1:4d} | R²={best_r2:.4f} | "
                  f"{lv_tokens_to_str(best_prog, best_cs)}")

    return best_prog, best_r2, best_cs, history


In [ ]:
# ─── Search for dx/dt  ───────────────────────────────────────────────────────
# X_clean, dxdt_c were prepared in Section 4's data prep cell.
xv_lv = X_clean[:, 0].astype(np.float32)
yv_lv = X_clean[:, 1].astype(np.float32)

print("=" * 60)
print("Searching for  dx/dt = alpha*x - beta*x*y")
print("Target prefix:  [-, *, c, x, *, c, *, x, y]  (9 tokens)")
print(f"Vocabulary: {LV_TOKENS}")
print("=" * 60)

prog_dx, r2_dx, cs_dx, hist_dx = train_lv_dso(
    xv_lv, yv_lv, dxdt_c.astype(np.float32),
    n_iter=800, batch_size=128)

print(f"\nFinal result for dx/dt:")
print(f"  Recovered: {lv_tokens_to_str(prog_dx, cs_dx)}")
print(f"  R² = {r2_dx:.4f}")
print(f"  True: alpha=1.0, beta=0.1  →  (1.0 * x) - (0.1 * (x * y))")


In [ ]:
# ─── Search for dy/dt ────────────────────────────────────────────────────────
print("=" * 60)
print("Searching for  dy/dt = delta*x*y - gamma*y")
print("Target prefix:  [-, *, c, *, x, y, *, c, y]  (9 tokens)")
print("=" * 60)

prog_dy, r2_dy, cs_dy, hist_dy = train_lv_dso(
    xv_lv, yv_lv, dydt_c.astype(np.float32),
    n_iter=800, batch_size=128)

print(f"\nFinal result for dy/dt:")
print(f"  Recovered: {lv_tokens_to_str(prog_dy, cs_dy)}")
print(f"  R² = {r2_dy:.4f}")
print(f"  True: delta=0.075, gamma=1.5  →  (0.075 * (x * y)) - (1.5 * y)")


In [ ]:
# ─── Convergence ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, hist, title in zip(axes, [hist_dx, hist_dy],
                            ['dx/dt  (prey equation)', 'dy/dt  (predator equation)']):
    ax.plot(hist)
    ax.set_xlabel('Iteration'); ax.set_ylabel('Best R² seen')
    ax.set_title(f'Extended DSO — {title}')
    ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()
print("Each jump in R² corresponds to the policy discovering a better expression structure.")


In [ ]:
# ─── Evaluate fitted expression at arbitrary (x, y) points ──────────────────
def eval_lv_fitted(tokens, xv, yv, c_vals):
    """Evaluate the expression tree with its fitted constant values."""
    if len(c_vals) == 0:
        return np.zeros_like(xv)
    total = np.zeros_like(xv, dtype=np.float64)
    for k, c in enumerate(c_vals):
        col = _eval_kth_const(tokens,
                              xv.astype(np.float64),
                              yv.astype(np.float64), k)
        if col is not None:
            total += float(c) * col
    return total


def make_dso_ode(prog_x, cs_x, prog_y, cs_y):
    """Wrap recovered expressions as a scipy ODE RHS."""
    def rhs(t, state):
        xv = np.array([state[0]]); yv = np.array([state[1]])
        dx = float(eval_lv_fitted(prog_x, xv, yv, cs_x)[0])
        dy = float(eval_lv_fitted(prog_y, xv, yv, cs_y)[0])
        return [dx, dy]
    return rhs


# Simulate using recovered equations
sol_rec = solve_ivp(
    make_dso_ode(prog_dx, cs_dx, prog_dy, cs_dy),
    (0, 40), y0=[10.0, 5.0],
    t_eval=t_eval, method='RK45', rtol=1e-6, atol=1e-6)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_traj, x_traj, label='Prey (true)',      lw=2, color='steelblue')
axes[0].plot(t_traj, y_traj, label='Predator (true)',  lw=2, color='darkorange')
if sol_rec.success:
    axes[0].plot(sol_rec.t, sol_rec.y[0], '--', lw=1.5,
                 label='Prey (recovered)',     color='steelblue',  alpha=0.75)
    axes[0].plot(sol_rec.t, sol_rec.y[1], '--', lw=1.5,
                 label='Predator (recovered)', color='darkorange', alpha=0.75)
axes[0].set_xlabel('Time'); axes[0].set_ylabel('Population')
axes[0].set_title('Trajectories: ground truth vs. recovered ODE')
axes[0].legend(fontsize=8)

axes[1].plot(x_traj, y_traj, lw=2,  label='True',      color='mediumseagreen')
if sol_rec.success:
    axes[1].plot(sol_rec.y[0], sol_rec.y[1], '--', lw=1.5,
                 label='Recovered', color='tomato')
axes[1].set_xlabel('Prey x'); axes[1].set_ylabel('Predator y')
axes[1].set_title('Phase portrait')
axes[1].legend()

plt.tight_layout(); plt.show()

print("Recovered equations:")
print(f"  dx/dt = {lv_tokens_to_str(prog_dx, cs_dx)}   (R²={r2_dx:.4f})")
print(f"  dy/dt = {lv_tokens_to_str(prog_dy, cs_dy)}   (R²={r2_dy:.4f})")
if not sol_rec.success:
    print("\nSimulation did not complete — try re-running the search cells.")


---
## 6. Symbolic Regression on Noisy Data

In practice we observe **noisy** state measurements, not exact trajectories.
Measurement noise has two consequences:

1. The state values $\tilde{x}(t_i) = x(t_i) + \epsilon_i$ are corrupted — but this is manageable.
2. **Derivative estimation is strongly amplified.** A central finite-difference estimate:
   $$\hat{\dot{x}}_i \approx \frac{\tilde{x}_{i+1} - \tilde{x}_{i-1}}{2\Delta t}$$
   divides by $\Delta t$, so high-frequency noise is amplified by a factor of $\sim 1/(2\Delta t)$.
   For our time grid ($\Delta t = 0.02$), noise is amplified roughly $25\times$ — a derivative
   estimate that looks smooth in the original signal can look like pure static after differencing.
   This is not a software bug; it is a fundamental property of numerical differentiation.

This is precisely the problem that **weak-form SINDy** was designed to solve: by integrating
against test functions rather than differentiating, the noise amplification is avoided entirely.

---

### A practical mitigation: the Savitzky–Golay filter

For expression-tree methods that work pointwise (like our mini-DSO), we still need a usable
derivative estimate from noisy data. The idea is simple: **smooth first, then differentiate.**

Instead of differencing adjacent noisy measurements, slide a window of $w$ time points
along the trajectory and fit a **local polynomial of degree $p$** to the points in that
window. Then differentiate that polynomial analytically and evaluate at the window's
center. This is the **Savitzky–Golay filter** (Savitzky & Golay 1964).

Concretely, at each point $t_i$ with window half-width $m = (w-1)/2$:
1. Fit $\hat{p}_i(t) = \sum_{k=0}^{p} c_k (t - t_i)^k$ to the $w$ surrounding points by least squares.
2. Set $\hat{\dot{x}}_i = \hat{p}_i'(t_i) = c_1$.

This is equivalent to a linear convolution filter with precomputed coefficients $h_j$:

$$\hat{\dot{x}}_i = \text{SG}_{w,p}^{(1)}(\tilde{x})_i = \sum_{j=-m}^{m} h_j \, \tilde{x}_{i+j}$$

> **Intuition:** a naive finite difference uses 2 neighboring points, so a single noisy
> measurement dominates. A local cubic with $w = 31$ uses 31 points and is far less sensitive
> to individual outliers — at the cost of slightly blurring rapid changes in the derivative.

The plot below shows how dramatically derivative quality degrades under noise, and how
much Savitzky–Golay smoothing helps. Even with smoothing, the recovered derivative is
imperfect. This is why weak-form SINDy remains the preferred approach when noise is
a serious concern in biological data: it avoids derivative estimation entirely by
working in integral form.


In [ ]:
rng = np.random.default_rng(0)
noise_level = 0.5   # std dev of additive Gaussian noise

x_noisy = x_traj + rng.normal(0, noise_level, size=x_traj.shape)
y_noisy = y_traj + rng.normal(0, noise_level, size=y_traj.shape)

dt = t_traj[1] - t_traj[0]

# Naive finite differences (central)
dxdt_fd = np.gradient(x_noisy, dt)
dydt_fd = np.gradient(y_noisy, dt)

# Savitzky–Golay smoothed derivatives
window, poly = 31, 3
dxdt_sg = savgol_filter(x_noisy, window_length=window, polyorder=poly, deriv=1, delta=dt)
dydt_sg = savgol_filter(y_noisy, window_length=window, polyorder=poly, deriv=1, delta=dt)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, true_d, fd_d, sg_d, title in zip(
        axes,
        [dxdt_true, dydt_true], [dxdt_fd, dydt_fd], [dxdt_sg, dydt_sg],
        ['dx/dt', 'dy/dt']):
    ax.plot(t_traj[:200], true_d[:200], label='True derivative', lw=2)
    ax.plot(t_traj[:200], fd_d[:200],  label='Finite diff (noisy)', alpha=0.5, lw=0.8)
    ax.plot(t_traj[:200], sg_d[:200],  label='Savitzky–Golay', linestyle='--', lw=1.5)
    ax.set_xlabel('Time'); ax.set_title(title); ax.legend(fontsize=8)
plt.suptitle(f'Derivative estimation with noise (σ = {noise_level})')
plt.tight_layout(); plt.show()


---
## 7. DSO vs. SINDy: When Does Each Shine?

Both produce interpretable symbolic ODEs. The choice depends on your data and prior knowledge.

**Use SINDy (including weak-form) when:**
- You have a reasonable idea of which functions to include in $\Theta$ (e.g., polynomial,
  trigonometric, exponential)
- You need fast, reliable recovery with mathematical guarantees (RIP conditions)
- Data are noisy: weak-form SINDy handles this better than any expression-tree method
  currently does by default
- You want a *convex* optimisation problem with a global solution

**Use expression-tree methods (DSO or PySR) when:**
- You do not know which functions govern the system — you want to *discover* the functional form
- The true dynamics involve nested/composed expressions unlikely to appear in a flat library
  (e.g., $\sin(x/y)$, $\exp(-x^2)$)
- You are willing to invest compute time in exchange for greater flexibility
- You want to **rank** a Pareto front of expressions trading off complexity and accuracy
  (PySR produces this automatically)

**An interesting middle ground:**
DSO and SINDy are not mutually exclusive. One emerging approach is to use DSO to
*discover candidate features* for a library, then use SINDy's sparse regression for
final identification — combining the flexibility of tree search with the noise robustness
of convex regression.


---
## (Optional) Appendix: SINDy on the Same Data

For comparison, here is a minimal PySINDy run on the same clean Lotka–Volterra data.
Since the audience knows SINDy, this is brief — just enough to see the output format
side by side with the PySR result above.

Run this section only if time permits.


In [ ]:
# !pip install pysindy -q

# import pysindy as ps
# from pysindy.differentiation import FiniteDifference

# # Build the state matrix: rows = time points, cols = [x, y]
# X_sindy = np.column_stack([x_traj[idx], y_traj[idx]])

# # Polynomial library up to degree 2
# library = ps.PolynomialLibrary(degree=2)

# model_sindy = ps.SINDy(
#     differentiation_method=FiniteDifference(),
#     feature_library=library,
#     optimizer=ps.STLSQ(threshold=0.05),
#     feature_names=['x', 'y']
# )
# model_sindy.fit(X_sindy, t=t_traj[idx])
# model_sindy.print()

print("Uncomment the cell above (and install pysindy) to run SINDy on the same data.")
print("Expected output: two equations with coefficients close to the true Lotka–Volterra parameters.")


---
## Summary

| | PySR / DSO | SINDy |
|--|-----------|-------|
| Representation | Symbolic expression tree | Sparse linear combination over library $\Theta$ |
| Search | Evolutionary (PySR) / RNN+RL (DSO) | Convex sparse regression (STLSQ/LASSO) |
| Library needed? | No | Yes |
| Nested expressions | Natural | Requires manual library extension |
| Noise robustness | Moderate (smoothing helps) | Excellent (weak-form) |
| Compute | Minutes | Seconds |
| Output | Pareto front of expressions | Single sparse model |

**Key takeaways:**
- DSO frames symbolic regression as a **reinforcement learning** problem: the RNN policy
  $\pi_\theta$ generates expression trees; REINFORCE maximises expected $R^2$.
- PySR achieves the same goal with evolutionary search — practical and robust.
- Both approaches eliminate the need for a pre-specified function library, at the cost of
  more compute and weaker theoretical guarantees.
- Noise in trajectory data is a fundamental challenge. Derivative amplification is
  unavoidable with finite differences; the Savitzky–Golay filter partially mitigates it.
  Weak-form SINDy eliminates the problem by design.
